In [1]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from pycbc.detector import Detector

import readligo
from readligo import apply_dq_mask, heterodyne_downsample, clean_narrowband

In [12]:
# Data
DATA_DIR = "/Users/saifa/Library/CloudStorage/OneDrive-weizmann.ac.il/O3a_marvin/"

# Band
F_BAND = (100, 101)
F_H = (F_BAND[0] + F_BAND[1]) / 2  # 100.5 Hz

# Signal
AMPLITUDE = 1e-24
SKY_POS = (0.3, 0.2)  # (ra, dec)
DETECTOR_NAME = "H1"
PSI = 0
N_SIGNALS = 500

In [13]:
hdf5_files = sorted(
    Path(DATA_DIR).glob("*.hdf5"),
    key=lambda p: int(p.stem.split('-')[-2])
)
filename = str(hdf5_files[4])

strain, time, channel_dict = readligo.loaddata(filename)
_, gps_start, *_ = readligo.read_hdf5(filename)
fs = 4096

clean_strain, dq_mask_1hz = apply_dq_mask(strain, channel_dict, fs)
# Heterodyne + decimate  =>  non-cleaned baseband
narrowband, fs_new = heterodyne_downsample(clean_strain, fs, F_BAND)

# signal
N = len(narrowband)
gps_times = gps_start + np.arange(N) / fs_new
df = 0.2 
t_ref = gps_times[N // 2]

det = Detector(DETECTOR_NAME)
fplus, fcross = det.antenna_pattern(SKY_POS[0], SKY_POS[1], PSI, gps_times)
signal = AMPLITUDE * (fplus + 1j * fcross) * np.exp(2j * np.pi * df * (gps_times - t_ref))

# Inject into the non-cleaned baseband data
narrowband_injected = narrowband + signal

cleaned_injected, sample_mask_injected, diag_injected = clean_narrowband(
    narrowband_injected, 
    dq_mask_1hz, 
    fs_new
)

In [38]:
def covariance_matrix(fplus, fcross):
    vp = np.dot(fplus, fplus)
    vc = np.dot(fcross, fcross)
    vpc = np.dot(fplus, fcross)
    return np.array([[vp, vpc], [vpc, vc]])


def data_projection_fourier(data, fplus, fcross, sample_mask=None):
    if sample_mask is not None:
        data = data[sample_mask]
        fplus = fplus[sample_mask]
        fcross = fcross[sample_mask]
        dplus = np.fft.fft(data * fplus)
        dcross = np.fft.fft(data * fcross)
        return np.stack([dplus, dcross], axis=0)
    else:
        dplus = np.fft.fft(data * fplus)
        dcross = np.fft.fft(data * fcross)
        return np.stack([dplus, dcross], axis=0)
    

In [39]:
# sigma_noncleaned = np.sqrt(np.mean(np.abs(narrowband)**2))
# sigma_cleaned = np.sqrt(np.mean(np.abs(cleaned_injected[sample_mask_injected])**2))

# print(f"Non-cleaned sigma (includes glitches): {sigma_noncleaned:.6e}")
# print(f"Cleaned sigma (quiet background only): {sigma_cleaned:.6e}")

# total_samples = len(sample_mask_injected)
# good_samples = np.sum(sample_mask_injected)
# print(f"Samples kept: {good_samples} / {total_samples}")
# print(f"Samples removed as artifacts: {total_samples - good_samples}")

In [41]:
# Precompute inverse covariance matrix
C_inv = np.linalg.inv(covariance_matrix(fplus, fcross))

sigma_cleaned = 4.159272e-24
sigma_noncleaned = 3.752088e-23
# Project the data into the Fourier domain and normalize by sigma
proj_cleaned = data_projection_fourier(cleaned_injected / sigma_cleaned, fplus, fcross, sample_mask_injected)
proj_noncleaned = data_projection_fourier(narrowband_injected / sigma_noncleaned, fplus, fcross, sample_mask=None)

In [42]:
# Get the FFT frequency bins
freqs = np.fft.fftfreq(N, d=1/fs_new)

# Find the exact index of our injected frequency (df = 0.2)
f_idx = np.argmin(np.abs(freqs - df))

# Calculate the score for the Cleaned data
d_k_cleaned = proj_cleaned[:, f_idx]
score_cleaned = np.real(d_k_cleaned.conj() @ C_inv @ d_k_cleaned)

# Calculate the score for the Non-Cleaned data
d_k_noncleaned = proj_noncleaned[:, f_idx]
score_noncleaned = np.real(d_k_noncleaned.conj() @ C_inv @ d_k_noncleaned)

print(f"Injected Frequency: {freqs[f_idx]:.4f} Hz")
print(f"Detection Score (Non-Cleaned): {score_noncleaned:.2f}")
print(f"Detection Score (Cleaned):     {score_cleaned:.2f}")

Injected Frequency: 0.2000 Hz
Detection Score (Non-Cleaned): 1.25
Detection Score (Cleaned):     2.84


In [43]:
sig_proj_noncleaned = data_projection_fourier(signal / sigma_noncleaned, fplus, fcross)
s_k_noncleaned = sig_proj_noncleaned[:, f_idx]
injected_score_noncleaned = np.real(s_k_noncleaned.conj() @ C_inv @ s_k_noncleaned)

print(f"Injected score (non-cleaned): {injected_score_noncleaned:.2f}")

Injected score (non-cleaned): 1.91
